# Data Cleaning and Preprocessing Project
## Renewable Hydrogen Production Dataset

**Project:** Clean and preprocess raw renewable hydrogen production data

**Dataset Source:** Kaggle - Renewable Hydrogen Data

**Student:** AI & Data Science Engineering

**Date:** November 2, 2025

---

## Objectives
1. Load and explore raw dataset
2. Identify and handle missing values
3. Detect and treat outliers
4. Handle duplicate records
5. Feature engineering and transformation
6. Data validation and quality checks
7. Save cleaned dataset for further analysis

## Step 1: Install Required Libraries and Setup

In [ ]:
# Install required libraries
!pip install opendatasets pandas numpy matplotlib seaborn scikit-learn -q

print("✓ Libraries installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Step 2: Download Dataset from Kaggle

In [ ]:
# Download dataset from Kaggle
import opendatasets as od

# Dataset URL
dataset_url = 'https://www.kaggle.com/datasets/mshabrawy/renewable-hydrogen-data'

# Download (you'll be prompted for Kaggle credentials)
print("Downloading dataset from Kaggle...")
print("You may be prompted to enter your Kaggle username and API key.")
print("Get your API key from: https://www.kaggle.com/me/account")
print("-" * 70)

od.download(dataset_url)

print("\n✓ Dataset downloaded successfully!")

## Step 3: Load Raw Dataset

In [ ]:
# Load the dataset
df_raw = pd.read_csv('renewable-hydrogen-data/Finall data (1).csv')

print("="*80)
print("RAW DATASET LOADED")
print("="*80)
print(f"\nDataset Shape: {df_raw.shape}")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

print("\nColumn Names:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Display first few rows
print("\nFirst 5 rows of RAW dataset:")
df_raw.head()

In [ ]:
# Dataset information
print("\nDataset Info:")
df_raw.info()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df_raw.describe()

## Step 4: Data Quality Assessment

In [ ]:
# Create a copy for cleaning
df = df_raw.copy()

print("="*80)
print("DATA QUALITY ASSESSMENT")
print("="*80)

# Check for missing values
print("\n1. MISSING VALUES:")
print("-" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("✓ No missing values found!")
else:
    print(f"\n⚠ Total missing values: {missing.sum():,}")

In [ ]:
# Check for duplicates
print("\n2. DUPLICATE RECORDS:")
print("-" * 60)
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates:,}")

if duplicates > 0:
    print(f"Percentage of duplicates: {(duplicates/len(df))*100:.2f}%")
else:
    print("✓ No duplicate records found!")

In [ ]:
# Check data types
print("\n3. DATA TYPES:")
print("-" * 60)
print(df.dtypes)

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")

## Step 5: Handle Missing Values

In [ ]:
print("="*80)
print("HANDLING MISSING VALUES")
print("="*80)

# Check if there are any missing values
if df.isnull().sum().sum() > 0:
    print("\nHandling missing values...")
    
    # For numerical columns: fill with median
    for col in numerical_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"✓ Filled {col} with median: {median_val:.2f}")
    
    # For categorical columns: fill with mode
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f"✓ Filled {col} with mode: {mode_val}")
    
    print(f"\n✓ All missing values handled!")
    print(f"Remaining missing values: {df.isnull().sum().sum()}")
else:
    print("\n✓ No missing values to handle!")

## Step 6: Handle Duplicate Records

In [ ]:
print("="*80)
print("HANDLING DUPLICATE RECORDS")
print("="*80)

initial_rows = len(df)
duplicates_before = df.duplicated().sum()

if duplicates_before > 0:
    print(f"\nFound {duplicates_before:,} duplicate rows")
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    final_rows = len(df)
    removed = initial_rows - final_rows
    
    print(f"✓ Removed {removed:,} duplicate rows")
    print(f"Final dataset size: {final_rows:,} rows")
else:
    print("\n✓ No duplicate records found!")
    print(f"Dataset size: {len(df):,} rows")

## Step 7: Outlier Detection and Treatment

In [ ]:
print("="*80)
print("OUTLIER DETECTION (IQR Method)")
print("="*80)

def detect_outliers_iqr(df, column):
    """Detect outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("\nOutlier Summary:")
print("-" * 70)
print(f"{'Column':<30} {'Outliers':<12} {'Lower Bound':<15} {'Upper Bound':<15}")
print("-" * 70)

outlier_summary = {}
for col in numerical_cols:
    n_outliers, lower, upper = detect_outliers_iqr(df, col)
    outlier_summary[col] = n_outliers
    print(f"{col:<30} {n_outliers:<12} {lower:<15.2f} {upper:<15.2f}")

total_outliers = sum(outlier_summary.values())
print("-" * 70)
print(f"Total outlier instances: {total_outliers:,}")

if total_outliers > 0:
    print("\n⚠ Outliers detected. These may represent:")
    print("  - Measurement errors")
    print("  - Extreme but valid conditions")
    print("  - Data entry errors")
    print("\nNote: Outliers are kept for renewable energy data as they may")
    print("represent valid extreme weather/operating conditions.")
else:
    print("\n✓ No outliers detected!")

In [ ]:
# Visualize outliers with box plots
print("\nVisualizing outliers...")

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols[:9]):
    axes[idx].boxplot(df[col].dropna(), vert=True)
    axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Value')
    axes[idx].grid(True, alpha=0.3)

# Hide extra subplots if less than 9 columns
for idx in range(len(numerical_cols), 9):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Box Plots for Outlier Detection', fontsize=14, fontweight='bold', y=1.002)
plt.show()

print("✓ Box plots generated!")

## Step 8: Data Validation and Quality Checks

In [ ]:
print("="*80)
print("DATA VALIDATION CHECKS")
print("="*80)

# Check for negative values in columns that should be positive
print("\n1. NEGATIVE VALUE CHECK:")
print("-" * 60)

for col in numerical_cols:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"⚠ {col}: {negative_count} negative values found")
    else:
        print(f"✓ {col}: No negative values")

# Check for zero variance columns
print("\n2. ZERO VARIANCE CHECK:")
print("-" * 60)

for col in numerical_cols:
    if df[col].std() == 0:
        print(f"⚠ {col}: Zero variance (constant column)")
    else:
        print(f"✓ {col}: Variance = {df[col].std():.4f}")

# Check value ranges
print("\n3. VALUE RANGE CHECK:")
print("-" * 60)
print(f"{'Column':<35} {'Min':<15} {'Max':<15}")
print("-" * 60)

for col in numerical_cols:
    print(f"{col:<35} {df[col].min():<15.2f} {df[col].max():<15.2f}")

## Step 9: Feature Engineering (Optional)

In [ ]:
print("="*80)
print("FEATURE ENGINEERING")
print("="*80)

# Create new features if applicable
# Example: Total Renewable Power
if 'PV_Power_kW' in df.columns and 'Wind_Power_kW' in df.columns:
    df['Total_Renewable_Power_kW'] = df['PV_Power_kW'] + df['Wind_Power_kW']
    print("✓ Created 'Total_Renewable_Power_kW' feature")

# Example: Renewable Power Ratio
if 'PV_Power_kW' in df.columns and 'Wind_Power_kW' in df.columns:
    df['PV_to_Wind_Ratio'] = df['PV_Power_kW'] / (df['Wind_Power_kW'] + 1)  # +1 to avoid division by zero
    print("✓ Created 'PV_to_Wind_Ratio' feature")

# Example: Hydrogen Production Efficiency
if 'Hydrogen_Production_kg/day' in df.columns and 'System_Efficiency_%' in df.columns:
    df['Efficiency_Factor'] = df['Hydrogen_Production_kg/day'] * df['System_Efficiency_%'] / 100
    print("✓ Created 'Efficiency_Factor' feature")

print(f"\nNew dataset shape: {df.shape}")
print(f"New features created: {df.shape[1] - df_raw.shape[1]}")

## Step 10: Data Distribution Analysis

In [ ]:
print("="*80)
print("DATA DISTRIBUTION ANALYSIS")
print("="*80)

# Select key numerical columns for visualization
key_cols = numerical_cols[:6]  # First 6 numerical columns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(key_cols):
    axes[idx].hist(df[col].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Distribution Plots', fontsize=14, fontweight='bold', y=1.002)
plt.show()

print("✓ Distribution plots generated!")

## Step 11: Correlation Analysis

In [ ]:
print("="*80)
print("CORRELATION ANALYSIS")
print("="*80)

# Calculate correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Correlation heatmap generated!")

# Find highly correlated pairs
print("\nHighly Correlated Feature Pairs (|r| > 0.7):")
print("-" * 60)

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            pair = (correlation_matrix.columns[i], 
                   correlation_matrix.columns[j], 
                   correlation_matrix.iloc[i, j])
            high_corr_pairs.append(pair)
            print(f"{pair[0]:<30} <-> {pair[1]:<30} : {pair[2]:.3f}")

if not high_corr_pairs:
    print("No highly correlated pairs found.")

## Step 12: Final Data Summary

In [ ]:
print("="*80)
print("FINAL CLEANED DATASET SUMMARY")
print("="*80)

print(f"\n📊 Dataset Dimensions:")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")

print(f"\n🔢 Data Types:")
print(f"   Numerical columns: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"   Categorical columns: {len(df.select_dtypes(include=['object']).columns)}")

print(f"\n✅ Data Quality:")
print(f"   Missing values: {df.isnull().sum().sum()}")
print(f"   Duplicate rows: {df.duplicated().sum()}")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n📈 Comparison:")
print(f"   Original rows: {df_raw.shape[0]:,}")
print(f"   Cleaned rows: {df.shape[0]:,}")
print(f"   Rows removed: {df_raw.shape[0] - df.shape[0]:,}")
print(f"   Data retention: {(df.shape[0]/df_raw.shape[0])*100:.2f}%")

In [ ]:
# Display cleaned dataset sample
print("\nCleaned Dataset - First 10 rows:")
df.head(10)

In [ ]:
# Final statistical summary
print("\nCleaned Dataset - Statistical Summary:")
df.describe()

## Step 13: Save Cleaned Dataset

In [ ]:
print("="*80)
print("SAVING CLEANED DATASET")
print("="*80)

# Save cleaned dataset
df.to_csv('Renewable_Hydrogen_Data_CLEANED.csv', index=False)
print("\n✓ Cleaned dataset saved as: 'Renewable_Hydrogen_Data_CLEANED.csv'")

# Also save the raw dataset for reference
df_raw.to_csv('Renewable_Hydrogen_Data_RAW.csv', index=False)
print("✓ Raw dataset saved as: 'Renewable_Hydrogen_Data_RAW.csv'")

print("\n" + "="*80)
print("DATA CLEANING AND PREPROCESSING COMPLETE!")
print("="*80)

print("\n📁 Files Generated:")
print("   1. Renewable_Hydrogen_Data_RAW.csv (original data)")
print("   2. Renewable_Hydrogen_Data_CLEANED.csv (cleaned data)")
print("   3. This Jupyter notebook (.ipynb)")

print("\n✅ Next Steps:")
print("   - Use cleaned dataset for machine learning models")
print("   - Perform exploratory data analysis (EDA)")
print("   - Build predictive models")
print("   - Feature selection and engineering")

## Step 14: Data Cleaning Report

In [ ]:
# Generate cleaning report
report = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    DATA CLEANING & PREPROCESSING REPORT                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

Dataset: Renewable Hydrogen Production Data
Source: Kaggle (mshabrawy/renewable-hydrogen-data)
Date: November 2, 2025

────────────────────────────────────────────────────────────────────────────────
CLEANING OPERATIONS PERFORMED:
────────────────────────────────────────────────────────────────────────────────

1. ✓ Missing Value Treatment
   - Numerical: Filled with median
   - Categorical: Filled with mode
   
2. ✓ Duplicate Record Removal
   - Identified and removed duplicate rows
   - Reset index after removal
   
3. ✓ Outlier Detection
   - Used IQR method for detection
   - Outliers retained (valid extreme conditions)
   
4. ✓ Data Type Validation
   - Verified appropriate data types
   - Ensured numerical/categorical consistency
   
5. ✓ Feature Engineering
   - Created derived features
   - Added ratio and efficiency metrics
   
6. ✓ Data Quality Checks
   - Negative value detection
   - Zero variance check
   - Range validation

────────────────────────────────────────────────────────────────────────────────
RESULTS:
────────────────────────────────────────────────────────────────────────────────

Original Dataset:
  • Rows: {df_raw.shape[0]:,}
  • Columns: {df_raw.shape[1]}
  • Missing Values: {df_raw.isnull().sum().sum()}
  • Duplicates: {df_raw.duplicated().sum()}

Cleaned Dataset:
  • Rows: {df.shape[0]:,}
  • Columns: {df.shape[1]}
  • Missing Values: {df.isnull().sum().sum()}
  • Duplicates: {df.duplicated().sum()}
  • Data Retention: {(df.shape[0]/df_raw.shape[0])*100:.2f}%

────────────────────────────────────────────────────────────────────────────────
RECOMMENDATIONS:
────────────────────────────────────────────────────────────────────────────────

1. The dataset is now ready for machine learning applications
2. Consider feature scaling/normalization for ML models
3. Use correlation analysis to identify redundant features
4. Perform train-test split before model training
5. Monitor for data drift in production

╚══════════════════════════════════════════════════════════════════════════════╝
"""

print(report)

# Save report to file
with open('Data_Cleaning_Report.txt', 'w') as f:
    f.write(report)

print("\n✓ Cleaning report saved as: 'Data_Cleaning_Report.txt'")